# Module `dynamic_costs.py`

Ce notebook explique la logique de cout dynamique appliquee a chaque vraie arete du graphe residuel.

Important : on modifie toujours les **vraies aretes**. La matrice completee par Dijkstra est ensuite reconstruite par `DynamicNetworkSimulator`. On ne tire donc jamais directement un cout dynamique entre deux sommets qui ne sont pas relies physiquement ; ce serait confondre decision de haut niveau et trajet reel.

Le module repond a une question precise : une route utilisable a un cout statique donne, comment faire varier son cout d'un tour a l'autre sans produire une simulation chaotique ou absurde ?


## 1. Les trois niveaux de cout

Chaque arete a trois couts qui coexistent :

| Niveau | Definition | Calcul |
|---|---|---|
| `base_cost` | Distance physique minimale | `math.dist(p_u, p_v)` |
| `static_cost` | Cout apres contrainte statique | `base_cost * (1 + static_surcharge)` ou `inf` si interdit |
| `dynamic_cost` | Cout courant au tour dynamique | tirage autour du cout precedent, borne par le statique |

Ces niveaux ne sont pas interchangeables.

- `base_cost` sert de plancher physique : une route ne devient pas magiquement plus courte que sa distance de base.
- `static_cost` encode les contraintes durables : travaux, surcharge recurrente, interdiction structurelle.
- `dynamic_cost` represente la variabilite de court terme : trafic, ralentissement temporaire, retour progressif a une situation normale.

Le solveur statique lit `completed_costs` construit depuis les `static_cost`. Le solveur dynamique lit `completed_costs` reconstruit depuis les `dynamic_cost` des aretes actuellement ON.


In [ ]:
import sys
from pathlib import Path
import random

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.dynamic_costs import (
    DEFAULT_DYNAMIC_SIGMA,
    dynamic_multiplier,
    initialize_dynamic_edge_costs,
    refresh_dynamic_edge_costs,
    sample_dynamic_edge_cost,
)
from cesipath.models import EdgeAttributes, EdgeStatus

## 2. La formule en detail

`sample_dynamic_edge_cost(edge, previous_cost, sigma, mean_reversion_strength, max_multiplier)` applique :

```text
baseline      = previous_cost si defini sinon static_cost
reverted_mean = baseline - mean_reversion_strength * (baseline - static_cost)
relative_gap  = max(0, baseline / static_cost - 1)
volatility    = max(0.35, 1 - 0.5 * min(relative_gap, 1))
sampled       = Normal(reverted_mean, static_cost * sigma * volatility)
result        = clamp(sampled, base_cost, static_cost * max_multiplier)
```

**Mean reversion** [1, 2]. Si le cout precedent est tres haut, la moyenne du prochain tirage est tiree vers `static_cost`. Cela evite les series qui explosent uniquement parce que le bruit a ete fort sur plusieurs tours. Si `mean_reversion_strength=0`, le cout suit davantage une marche aleatoire. Si la valeur vaut `1`, la moyenne revient directement au statique a chaque tour.

**Volatilite reduite quand le cout s'emballe.** Plus le cout courant est loin au-dessus du statique, plus l'ecart-type est compresse. C'est une seconde securite contre les trajectoires extremes.

**Bornes dures.** Le cout final ne descend pas sous `base_cost` et ne depasse pas `static_cost * max_multiplier`. Les solveurs voient ainsi des valeurs plausibles et comparables d'un tour a l'autre.


In [ ]:
edge = EdgeAttributes(base_cost=10.0, status=EdgeStatus.SURCHARGED, static_surcharge=0.25)
print("base_cost   :", edge.base_cost)
print("static_cost :", edge.static_cost)
print("max autorise :", edge.static_cost * 1.8)

## 3. Evolution sur 100 tours (plot)

On simule une seule arete sur 100 tours. Le trace montre trois phenomenes :

- le cout bouge, donc la dynamique a bien un effet ;
- quand le bruit l'eloigne du statique, la mean reversion le ramene progressivement ;
- le plafond `static_cost * max_multiplier` empeche les pics aberrants.

Cette cellule est surtout pedagogique. Dans la vraie simulation, le meme mecanisme est applique independamment a toutes les aretes actives du graphe residuel, avec un `random.Random(seed)` partage pour garder la reproductibilite.


In [ ]:
import matplotlib.pyplot as plt

rng = random.Random(42)
history = [edge.static_cost]
current = edge.static_cost
for _ in range(100):
    current = sample_dynamic_edge_cost(
        edge,
        previous_cost=current,
        rng=rng,
        sigma=0.18,
        mean_reversion_strength=0.35,
        max_multiplier=1.8,
    )
    history.append(current)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history, label="cout dynamique", color="#3498db")
ax.axhline(edge.base_cost, color="#2ecc71", linestyle="--", label="base_cost (plancher)")
ax.axhline(edge.static_cost, color="#9b59b6", linestyle="--", label="static_cost (centre du retour)")
ax.axhline(edge.static_cost * 1.8, color="#e74c3c", linestyle="--", label="plafond (x 1.8)")
ax.set_xlabel("tour")
ax.set_ylabel("cout")
ax.set_title("Evolution d'un cout dynamique sur 100 tours")
ax.legend(loc="best")
ax.grid(True, linestyle=":", alpha=0.5)
plt.show()

## 4. `dynamic_multiplier` : lecture du coefficient

`dynamic_multiplier(edge, dynamic_cost)` retourne :

```text
dynamic_cost / static_cost
```

Ce coefficient est plus parlant qu'un cout brut quand les aretes ont des longueurs tres differentes. Une hausse de `+5` n'a pas la meme signification sur une petite arete de cout `8` et sur une grande arete de cout `80`.

Usages typiques :

- afficher les aretes les plus perturbees dans une interface ;
- construire une jauge de congestion ;
- comparer la severite relative des variations dynamiques ;
- debugger le generateur en verifiant que les couts restent sous `max_multiplier`.

Pour une arete interdite (`static_cost = inf`) ou un cout statique nul, la fonction renvoie `inf`, car le ratio n'a pas de sens exploitable.


In [ ]:
for offset in [-2, 0, 2, 5]:
    cost = edge.static_cost + offset
    print(f"cout={cost:>6.2f}  coefficient={dynamic_multiplier(edge, cost)}")

## 5. `initialize_dynamic_edge_costs` et `refresh_dynamic_edge_costs`

Deux helpers permettent de travailler sur tout un dictionnaire d'aretes :

- `initialize_dynamic_edge_costs(edges)` : fixe chaque cout dynamique au `static_cost` correspondant ;
- `refresh_dynamic_edge_costs(edges, previous_costs, rng, ...)` : tire un nouveau cout pour chaque arete non interdite.

Les aretes `FORBIDDEN` sont ignorees. Elles n'appartiennent pas au reseau residuel exploitable, donc il serait inutile de leur donner une trajectoire dynamique.

**Pourquoi garder ces helpers hors du simulateur ?** Parce que le calcul du cout et la logique ON/OFF sont deux responsabilites differentes, conformement au principe de decomposition modulaire [3]. `dynamic_costs.py` sait faire varier une route. `dynamic_network.py` decide quelles routes sont disponibles et quand recalculer Dijkstra. Cette separation rend chaque morceau testable isolement.

**Piege de reproductibilite.** Si aucun `rng` n'est fourni, la fonction cree un `random.Random()` local. C'est pratique pour un usage ponctuel, mais les simulations comparatives doivent toujours passer un generateur explicite pour reproduire les memes trajectoires.


In [ ]:
edges = {
    (0, 1): EdgeAttributes(base_cost=10.0, status=EdgeStatus.SURCHARGED, static_surcharge=0.25),
    (1, 2): EdgeAttributes(base_cost=8.0),
    (0, 2): EdgeAttributes(base_cost=6.0, status=EdgeStatus.FORBIDDEN),
}

network_rng = random.Random(7)
current_costs = initialize_dynamic_edge_costs(edges)
print("Etat initial (FORBIDDEN exclue) :", current_costs)

current_costs = refresh_dynamic_edge_costs(
    edges,
    current_costs,
    rng=network_rng,
    sigma=0.18,
    mean_reversion_strength=0.35,
    max_multiplier=1.8,
)
print("Apres un tour                   :", current_costs)

## 6. Pourquoi ce modele ?

Trois objectifs conjoints :

- **Realisme** : les couts oscillent de facon continue, comme un trafic routier reel. On evite les sauts totalement independants d'un tour au suivant.
- **Securite mathematique** : les bornes dures empechent les solveurs de voir des valeurs aberrantes ou negatives.
- **Controle pedagogique** : les trois parametres (`sigma`, `mean_reversion_strength`, `max_multiplier`) ont une interpretation simple.

Le modele reste volontairement local : une arete ne depend pas de ses voisines. Ce n'est pas une simulation de trafic microscopique, mais c'est suffisant pour tester des strategies de re-optimisation dynamique. Le but du projet est de comparer les solveurs VRP dans un environnement mouvant, pas de modeliser finement les embouteillages.

A retenir : `dynamic_costs.py` produit des couts plausibles et bornes ; `dynamic_network.py` decide ensuite comment ces couts transforment la matrice complete que les algorithmes consomment.

---

## References

[1] **Uhlenbeck, G. E. & Ornstein, L. S. (1930).** *On the theory of the Brownian motion.* Physical Review, 36(5), 823-841. https://doi.org/10.1103/PhysRev.36.823

[2] **Ross, S. M. (2014).** *Introduction to Probability Models* (11th ed.). Academic Press.

[3] **Parnas, D. L. (1972).** *On the criteria to be used in decomposing systems into modules.* Communications of the ACM, 15(12), 1053-1058. https://doi.org/10.1145/361598.361623
